# 割引クーポンキャンペーンの最適化

OCR の説明に沿って、データ確認、2種類のモデリング、公平性と投資対効果の確認を行う。

## データの確認

In [ ]:
# このnotebookはリポジトリ直下から uv run jupyter notebook で開く想定
# sandbox環境では例: UV_CACHE_DIR=.uv-cache uv run jupyter notebook

import time
import pandas as pd
import pulp
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


In [ ]:
cust_df = pd.read_csv('customers.csv')
cust_df.shape


In [ ]:
# 先頭だけ確認
cust_df.head()


In [ ]:
cust_df.dtypes


In [ ]:
# 年齢区分と昨年度来店回数区分の分布
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
cust_df['age_cat'].hist(ax=axes[0])
axes[0].set_title('age_cat')
cust_df['freq_cat'].hist(ax=axes[1])
axes[1].set_title('freq_cat')
plt.tight_layout()
plt.show()


In [ ]:
age_order = ['age~19', 'age20~34', 'age35~49', 'age50~']

cust_pivot_df = pd.pivot_table(
    data=cust_df,
    values='customer_id',
    columns='freq_cat',
    index='age_cat',
    aggfunc='count'
).reindex(age_order)

cust_pivot_df


In [ ]:
sns.heatmap(cust_pivot_df, annot=True, fmt='d', cmap='Blues')
plt.show()


In [ ]:
prob_df = pd.read_csv('visit_probability.csv')
prob_df.shape


In [ ]:
prob_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 3))
for j, dm_col in enumerate(['prob_dm1', 'prob_dm2', 'prob_dm3']):
    table = pd.pivot_table(
        data=prob_df,
        values=dm_col,
        columns='freq_cat',
        index='age_cat',
        aggfunc='mean'
    ).reindex(age_order)
    sns.heatmap(table, annot=True, fmt='.1%', cmap='Blues', vmin=0, vmax=1, ax=axes[j])
    axes[j].set_title(dm_col)
plt.show()


## モデリング1: 会員個別送付モデル

In [ ]:
problem1 = pulp.LpProblem(name='DiscountCouponProblem1', sense=pulp.LpMaximize)

I = cust_df['customer_id'].to_list()
M = [1, 2, 3]

# x[i, m] = 会員iへDMパターンmを送るなら1
xim = {}
for i in I:
    for m in M:
        xim[i, m] = pulp.LpVariable(name=f'xim({i},{m})', cat='Binary')

len(xim)


In [ ]:
# 各会員に送るDMはいずれか1つ
for i in I:
    problem1 += pulp.lpSum(xim[i, m] for m in M) == 1


In [ ]:
keys = ['age_cat', 'freq_cat']
cust_prob_df = pd.merge(cust_df, prob_df, on=keys)

cust_prob_ver_df = (
    cust_prob_df
    .rename(columns={'prob_dm1': 1, 'prob_dm2': 2, 'prob_dm3': 3})
    .melt(id_vars=['customer_id'], value_vars=[1, 2, 3], var_name='dm', value_name='prob')
)
Pim = cust_prob_ver_df.set_index(['customer_id', 'dm'])['prob'].to_dict()
Pim[1, 1]


In [ ]:
# クーポン付与による来客増加数を最大化する
problem1 += pulp.lpSum((Pim[i, m] - Pim[i, 1]) * xim[i, m] for i in I for m in [2, 3])

Cm = {1: 0, 2: 1000, 3: 2000}

# 予算消費期待値は100万円以下
problem1 += pulp.lpSum(Cm[m] * Pim[i, m] * xim[i, m] for i in I for m in [2, 3]) <= 1_000_000


In [ ]:
S = prob_df['segment_id'].to_list()
Ns = cust_prob_df.groupby('segment_id')['customer_id'].count().to_dict()
Si = cust_prob_df.set_index('customer_id')['segment_id'].to_dict()

# 各セグメントで各パターンを10%以上送付する
for s in S:
    for m in M:
        problem1 += pulp.lpSum(xim[i, m] for i in I if Si[i] == s) >= 0.1 * Ns[s]


In [ ]:
start = time.time()
solver = pulp.PULP_CBC_CMD(gapRel=1e-3, msg=False)
status = problem1.solve(solver)
elapsed = time.time() - start
print(f'ステータス: {pulp.LpStatus[status]}')
print(f'目的関数値: {pulp.value(problem1.objective):.4}')
print(f'計算時間: {elapsed:.3f} 秒')


In [ ]:
send_dm_df = pd.DataFrame(
    [[xim[i, m].value() for m in M] for i in I],
    columns=['send_dm1', 'send_dm2', 'send_dm3']
)
cust_send_df = pd.concat([cust_df[['customer_id', 'age_cat', 'freq_cat']], send_dm_df], axis=1)
cust_send_df.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 3))
for j, dm_col in enumerate(['send_dm1', 'send_dm2', 'send_dm3']):
    table = pd.pivot_table(cust_send_df, values=dm_col, columns='freq_cat', index='age_cat', aggfunc='mean').reindex(age_order)
    sns.heatmap(table, annot=True, fmt='.1%', cmap='Blues', vmin=0, vmax=1, ax=axes[j])
    axes[j].set_title(f'{dm_col}_rate')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 3))
for j, dm_col in enumerate(['send_dm1', 'send_dm2', 'send_dm3']):
    table = pd.pivot_table(cust_send_df, values=dm_col, columns='freq_cat', index='age_cat', aggfunc='sum').reindex(age_order)
    sns.heatmap(table, annot=True, fmt='.1f', cmap='Blues', vmax=800, ax=axes[j])
    axes[j].set_title(f'{dm_col}_num')
plt.show()


## モデリング2: セグメント送付モデル

In [ ]:
problem2 = pulp.LpProblem(name='DiscountCouponProblem2', sense=pulp.LpMaximize)

# x[s, m] = セグメントsへのDMパターンmの送付率
xsm = {}
for s in S:
    for m in M:
        xsm[s, m] = pulp.LpVariable(name=f'xsm({s},{m})', lowBound=0, upBound=1, cat='Continuous')

for s in S:
    problem2 += pulp.lpSum(xsm[s, m] for m in M) == 1


In [ ]:
prob_ver_df = (
    prob_df
    .rename(columns={'prob_dm1': 1, 'prob_dm2': 2, 'prob_dm3': 3})
    .melt(id_vars=['segment_id'], value_vars=[1, 2, 3], var_name='dm', value_name='prob')
)
Psm = prob_ver_df.set_index(['segment_id', 'dm'])['prob'].to_dict()

problem2 += pulp.lpSum(Ns[s] * (Psm[s, m] - Psm[s, 1]) * xsm[s, m] for s in S for m in [2, 3])
problem2 += pulp.lpSum(Cm[m] * Ns[s] * Psm[s, m] * xsm[s, m] for s in S for m in [2, 3]) <= 1_000_000

for s in S:
    for m in M:
        problem2 += xsm[s, m] >= 0.1


In [ ]:
start = time.time()
status = problem2.solve(pulp.PULP_CBC_CMD(msg=False))
elapsed = time.time() - start
print(f'ステータス: {pulp.LpStatus[status]}')
print(f'目的関数値: {pulp.value(problem2.objective):.4}')
print(f'計算時間: {elapsed:.3f} 秒')


In [ ]:
seg_send_df = pd.concat([
    prob_df[['segment_id', 'age_cat', 'freq_cat']],
    pd.DataFrame([[xsm[s, m].value() for m in M] for s in S], columns=['send_prob_dm1', 'send_prob_dm2', 'send_prob_dm3'])
], axis=1)

fig, axes = plt.subplots(1, 3, figsize=(20, 3))
for j, dm_col in enumerate(['send_prob_dm1', 'send_prob_dm2', 'send_prob_dm3']):
    table = pd.pivot_table(seg_send_df, values=dm_col, columns='freq_cat', index='age_cat', aggfunc='mean').reindex(age_order)
    sns.heatmap(table, annot=True, fmt='.1%', cmap='Blues', vmin=0, vmax=1, ax=axes[j])
    axes[j].set_title(dm_col)
plt.show()


In [ ]:
seg_send_df['num_cust'] = seg_send_df['segment_id'].map(Ns)
for m in M:
    seg_send_df[f'send_num_dm{m}'] = seg_send_df[f'send_prob_dm{m}'] * seg_send_df['num_cust']

seg_send_df[['segment_id', 'send_num_dm1', 'send_num_dm2', 'send_num_dm3']].head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 3))
for j, dm_col in enumerate(['send_num_dm1', 'send_num_dm2', 'send_num_dm3']):
    table = pd.pivot_table(seg_send_df, values=dm_col, columns='freq_cat', index='age_cat').reindex(age_order)
    sns.heatmap(table, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=800, ax=axes[j])
    axes[j].set_title(dm_col)
plt.show()


## 結果の評価

### モデリング3: 送付率下限値最大化モデル

In [ ]:
problem3 = pulp.LpProblem(name='DiscountCouponProblem3', sense=pulp.LpMaximize)

xsm = {}
for s in S:
    for m in M:
        xsm[s, m] = pulp.LpVariable(name=f'xsm({s},{m})', lowBound=0, upBound=1, cat='Continuous')

y = pulp.LpVariable(name='y', lowBound=0, upBound=1, cat='Continuous')

problem3 += y

for s in S:
    problem3 += pulp.lpSum(xsm[s, m] for m in M) == 1
    for m in M:
        problem3 += xsm[s, m] >= y

problem3 += pulp.lpSum(Cm[m] * Ns[s] * Psm[s, m] * xsm[s, m] for s in S for m in [2, 3]) <= 1_000_000

status = problem3.solve(pulp.PULP_CBC_CMD(msg=False))
max_lowerbound = pulp.value(problem3.objective)
print(f'ステータス: {pulp.LpStatus[status]}, 目的関数値: {max_lowerbound:.3f}')


In [ ]:
fair_send_df = pd.concat([
    prob_df[['segment_id', 'age_cat', 'freq_cat']],
    pd.DataFrame([[xsm[s, m].value() for m in M] for s in S], columns=['send_prob_dm1', 'send_prob_dm2', 'send_prob_dm3'])
], axis=1)

fig, axes = plt.subplots(1, 3, figsize=(20, 3))
for j, dm_col in enumerate(['send_prob_dm1', 'send_prob_dm2', 'send_prob_dm3']):
    table = pd.pivot_table(fair_send_df, values=dm_col, columns='freq_cat', index='age_cat', aggfunc='mean').reindex(age_order)
    sns.heatmap(table, annot=True, fmt='.1%', cmap='Blues', vmin=0, vmax=1, ax=axes[j])
    axes[j].set_title(dm_col)
plt.show()


In [ ]:
# 最大化した下限値を固定して、来客増加数がどのくらい下がるか確認する
problem3b = pulp.LpProblem(name='DiscountCouponProblem3b', sense=pulp.LpMaximize)

xsm = {}
for s in S:
    for m in M:
        xsm[s, m] = pulp.LpVariable(name=f'xsm2({s},{m})', lowBound=0, upBound=1, cat='Continuous')

for s in S:
    problem3b += pulp.lpSum(xsm[s, m] for m in M) == 1
    for m in M:
        problem3b += xsm[s, m] >= max_lowerbound

problem3b += pulp.lpSum(Ns[s] * (Psm[s, m] - Psm[s, 1]) * xsm[s, m] for s in S for m in [2, 3])
problem3b += pulp.lpSum(Cm[m] * Ns[s] * Psm[s, m] * xsm[s, m] for s in S for m in [2, 3]) <= 1_000_000

status = problem3b.solve(pulp.PULP_CBC_CMD(msg=False))
print(f'ステータス: {pulp.LpStatus[status]}, 目的関数値: {pulp.value(problem3b.objective):.4}')


### 投資対効果の評価

In [ ]:
cost_list = []
cpa_list = []
inc_action_list = []

print('ステータス, キャンペーン費用, 来客増加数, CPA')
for cost in range(761850, 3_000_000, 100_000):
    tmp = pulp.LpProblem(name='DiscountCouponProblem2', sense=pulp.LpMaximize)
    xsm = {}
    for s in S:
        for m in M:
            xsm[s, m] = pulp.LpVariable(name=f'xsm_cost({cost},{s},{m})', lowBound=0, upBound=1, cat='Continuous')

    for s in S:
        tmp += pulp.lpSum(xsm[s, m] for m in M) == 1
        for m in M:
            tmp += xsm[s, m] >= 0.1

    tmp += pulp.lpSum(Ns[s] * (Psm[s, m] - Psm[s, 1]) * xsm[s, m] for s in S for m in [2, 3])
    tmp += pulp.lpSum(Cm[m] * Ns[s] * Psm[s, m] * xsm[s, m] for s in S for m in [2, 3]) <= cost

    status = tmp.solve(pulp.PULP_CBC_CMD(msg=False))
    inc_action = pulp.value(tmp.objective)
    cpa = cost / inc_action
    cost_list.append(cost)
    inc_action_list.append(inc_action)
    cpa_list.append(cpa)
    print(f'{pulp.LpStatus[status]}, {cost}, {inc_action:.4f}, {cpa:.5f}')


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.scatter(cost_list, inc_action_list, marker='x', label='Incremental visitor')
ax2.scatter(cost_list, cpa_list, label='CPA')

ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x:,.0f}'))
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x:,.0f}'))
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x:,.0f}'))
ax1.set_xlabel('Cost')
ax1.set_ylabel('Incremental visitor')
ax2.set_ylabel('CPA')

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, loc='upper center')
plt.show()
